In [ ]:
%pip install pandas
%pip install transformers
%pip install torch
%pip install datasets
%pip install 'math-verify[antlr4_13_2]' # pin as per math-verify install instr.
# restart the kernel after installing in your virtual environment to ensure the new packages are loaded

In [ ]:
import pandas as pd

## Grab our dataset of historical AIME problems

In [ ]:

from datasets import load_dataset

ds = load_dataset("gneubig/aime-1983-2024")

# get all features
filtered = ds['train']

# Save to csv
filtered.to_csv('aime_historical.csv')

df = pd.read_csv('./aime_historical.csv')
display(df.tail(50))

## Testing

Let's do a small test to see if our thinking traces works, we're calling HF correctly, etc.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_id = "deepseek-ai/DeepSeek-R1-Distill-Qwen-7B"  # pull from HuggingFace
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, output_hidden_states=True)

inputs = tokenizer("Why is the sky blue?", return_tensors="pt")

with torch.no_grad():
    outputs = model(**inputs, output_hidden_states=True)

# outputs.hidden_states is a tuple: one tensor per layer
# each tensor shape: (batch, seq_len, hidden_dim)
hidden_states = outputs.hidden_states

for layer_idx, layer_hs in enumerate(hidden_states):
    print(f"Layer {layer_idx}: {layer_hs.shape}")
    # e.g. Layer 0: torch.Size([1, 8, 2048])

# store as list of tensors per token across all layers:
# shape: [num_layers][batch, token, hidden_dim]
all_layers = [hs.squeeze(0) for hs in hidden_states]  # drop batch dim
# all_layers[layer][token] → hidden vector for that token at that layer

## Probe Training

Train a linear probe on hidden states collected at every CoT token. Binary label = 1 if the model's final answer matched the ground truth, 0 otherwise.

In [ ]:
# === FEATURE FLAG ===
DEBUG = True  # Set to False for full 1000-problem training run

MAX_PROBLEMS    = 3       if DEBUG else 1000
PROBE_EPOCHS    = 5       if DEBUG else 30
PROBE_LR        = 2e-3    if DEBUG else 5e-4
MAX_NEW_TOKENS  = 16384   # to prevent OOM/recursion issues.
PROBE_LAYER     = -1      # which hidden_states layer to probe (-1 = last, i.e. layer 28)
HIDDEN_DIM      = 3584    # DeepSeek-R1-Distill-Qwen-7B hidden dimension

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_id = "deepseek-ai/DeepSeek-R1-Distill-Qwen-7B"
device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    device_map="auto" if device == "cuda" else None,
)
model.eval()
print(f"Model loaded on {device}")

In [ ]:
import pandas as pd

df = pd.read_csv('./aime_historical.csv')
problems = df[['Question', 'Answer']].dropna().reset_index(drop=True)
print(f"Total AIME problems available: {len(problems)}")
problems.head()

In [ ]:
def build_prompt(problem_text: str) -> str:
    messages = [
        {
            "role": "user",
            "content": (
                "Solve the following AIME problem. "
                "Your final answer must be a non-negative integer (0-999), "
                "placed inside \\boxed{}.\n\n"
                + problem_text
            ),
        }
    ]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )


def _find_subseq(seq, sub):
    n, m = len(seq), len(sub)
    for i in range(n - m + 1):
        if seq[i : i + m] == sub:
            return i
    return -1


def get_cot_hidden_states(prompt: str):
    """Generate a response for `prompt`, extract CoT token hidden states.

    Returns:
        cot_hidden: FloatTensor [num_cot_tokens, HIDDEN_DIM] at PROBE_LAYER
        generated_text: str  (decoded generated tokens, special tokens included)
    """
    input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device)
    prompt_len = input_ids.shape[1]

    with torch.no_grad():
        generation = model.generate(
            input_ids,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            return_dict_in_generate=True,
            output_hidden_states=True,
        )

    gen_ids = generation.sequences
    generated_ids = gen_ids[0, prompt_len:]

    if generation.hidden_states:
        hs = torch.cat(
            [step_hidden[PROBE_LAYER][:, -1, :] for step_hidden in generation.hidden_states],
            dim=0,
        ).float().cpu()
    else:
        hs = torch.empty((0, HIDDEN_DIM), dtype=torch.float32)

    generated_text = tokenizer.decode(generated_ids, skip_special_tokens=False)

    # locate <think> / </think> in the generated token stream
    think_toks = tokenizer.encode("<think>",  add_special_tokens=False)
    end_toks   = tokenizer.encode("</think>", add_special_tokens=False)
    gen_list   = generated_ids.tolist()

    think_pos = _find_subseq(gen_list, think_toks)
    end_pos   = _find_subseq(gen_list, end_toks)

    if think_pos != -1 and end_pos != -1 and end_pos > think_pos:
        cot_start = think_pos + len(think_toks)
        cot_end   = end_pos
    else:
        # no explicit markers — treat all generated tokens as CoT
        cot_start = 0
        cot_end   = generated_ids.shape[0]

    cot_hidden = hs[cot_start:cot_end]  # [num_cot_tokens, HIDDEN_DIM]
    return cot_hidden, generated_text

In [ ]:
from tqdm.auto import tqdm
from math_verify import parse, verify

X_list, y_list = [], []
results = []

subset = problems.iloc[:MAX_PROBLEMS]

for idx, row in tqdm(subset.iterrows(), total=len(subset), desc="Collecting hidden states"):
    problem_text = row["Question"]
    correct_ans  = str(row["Answer"])

    prompt = build_prompt(problem_text)
    cot_hidden, gen_text = get_cot_hidden_states(prompt)

    gold       = parse(f"${correct_ans}$")
    predicted  = parse(gen_text)
    is_correct = bool(verify(gold, predicted))
    label      = 1 if is_correct else 0

    predicted_str = str(predicted[0]) if predicted else "<none>"

    results.append({
        "idx":        idx,
        "correct":    correct_ans,
        "predicted":  predicted_str,
        "label":      label,
        "cot_tokens": cot_hidden.shape[0],
    })

    if cot_hidden.shape[0] > 0:
        X_list.append(cot_hidden)
        y_list.extend([label] * cot_hidden.shape[0])

    print(f"  Problem {idx}: correct={correct_ans}, predicted={predicted_str}, "
          f"label={label}, cot_tokens={cot_hidden.shape[0]}")

results_df = pd.DataFrame(results)
print("\n", results_df)
print(f"\nTotal CoT tokens collected : {sum(x.shape[0] for x in X_list)}")
print(f"Label distribution          : {sum(y_list)} correct / "
      f"{len(y_list) - sum(y_list)} incorrect")

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

X = torch.cat(X_list, dim=0)                           # [N_tokens, HIDDEN_DIM]
y = torch.tensor(y_list, dtype=torch.float32)          # [N_tokens]

print(f"X: {X.shape}  y: {y.shape}")

dataset = TensorDataset(X, y)
loader  = DataLoader(dataset, batch_size=256, shuffle=True)

In [ ]:
import torch.nn as nn

class LinearProbe(nn.Module):
    def __init__(self, hidden_dim: int):
        super().__init__()
        self.fc = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        return self.fc(x).squeeze(-1)  # [batch]


probe     = LinearProbe(HIDDEN_DIM).to(device)
optimizer = torch.optim.Adam(probe.parameters(), lr=PROBE_LR)
criterion = nn.BCEWithLogitsLoss()
print(probe)

In [ ]:
for epoch in range(1, PROBE_EPOCHS + 1):
    probe.train()
    total_loss, n_correct, n_total = 0.0, 0, 0

    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        logits = probe(xb)
        loss   = criterion(logits, yb)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * len(yb)
        n_correct  += ((logits > 0).long() == yb.long()).sum().item()
        n_total    += len(yb)

    print(f"Epoch {epoch:3d}/{PROBE_EPOCHS}  "
          f"loss={total_loss / n_total:.4f}  "
          f"acc={n_correct / n_total:.4f}")

In [ ]:
torch.save(probe.state_dict(), "linear_probe.pt")
print("Saved probe to linear_probe.pt")